#### Import Libraries

In [17]:
# PDF Loader
from langchain_community.document_loaders import PyPDFLoader

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter 

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector Store
from langchain_community.vectorstores import FAISS

# Prompt Template
from langchain_core.prompts import PromptTemplate

# Gemini
import google.generativeai as genai

# Environment Variables
from dotenv import load_dotenv

# Python Utilities
import os

#### Load API Key

In [18]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

print("API Loaded Successfully")

API Loaded Successfully


#### Configure Gemini

In [19]:
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")
print("Gemini Connected")

Gemini Connected


#### Load PDF

In [20]:
pdf_path = (r"F:\GenAI\datasets\1.Introduction to Deep Learning.pdf") 
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print("Pages Loaded:", len(documents))

Pages Loaded: 6


#### Check First Page

In [21]:
print(documents[0].page_content[:1000])

What is Deep Learning? 
Deep learning is a subfield of AI and Machine Learning that is inspired by the structure of a human 
brain. 
Deep Learning does the same work as machine learning — that is, finding relationships between 
input and output and making predictions from data — but it goes a step further. While machine 
learning relies on statistical techniques to identify these relationships, deep learning uses a 
predefined logical structure called neural networks to automatically learn complex patterns from 
data.


#### Chunking

In [22]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
chunks = splitter.split_documents(documents)
print("Total Chunks:", len(chunks))

Total Chunks: 23


#### Create Embedding Model

In [23]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding Model Ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Model Ready


#### Create FAISS Vector Store

In [24]:
vector_store = FAISS.from_documents(chunks,embedding_model)
print("FAISS Created")

FAISS Created


#### Save FAISS

In [25]:
vector_store.save_local("faiss_index")
print("FAISS Saved")

FAISS Saved


#### Create Retriever

In [26]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})
print("Retriever Ready")

Retriever Ready


#### User Query

In [27]:
query = "What is RNN?"

#### Retrieve Relevant Chunks

In [28]:
docs = retriever.invoke(query)
print("Retrieved Chunks:", len(docs))

Retrieved Chunks: 3


#### Build Context

In [29]:
context = "\n\n".join([doc.page_content for doc in docs])
print(context[:2000])

such as image recognition, object detection and more. 
 
3.Recurrent Neural Network (RNN) 
It is a type of neural network designed to handle sequential data like text, time series or speech. 
Unlike regular networks, RNNs have loops that allow them to carry information from previous steps 
to the current step, making them good at learning from context.  
They are commonly used in language modelling, sentiment analysis, and speech recognition. 
 
4.LSTM

4.LSTM 
LSTM (Long Short-Term Memory) is a special type of RNN designed to remember information for long 
periods and avoid the vanishing gradient problem. 
 LSTMs are widely used in applications like machine translation, text generation, and time series 
forecasting. 
 
5.Auto encoders 
It is a type of neural network that compresses input data into smaller more efficient representation 
while preserving the most important information. It has two parts encoder that compresses the

Neural Networks (CNNs) are effective for image-related t

#### Create PromptTemplate

In [30]:
prompt_template = PromptTemplate.from_template(
"""
You are an AI Tutor.
Answer ONLY from the provided context.
Context:
{context}

Question:
{question}
"""
)

#### Format Prompt

In [31]:
final_prompt = prompt_template.format(context=context,question=query)
print(final_prompt[:2000])


You are an AI Tutor.
Answer ONLY from the provided context.
Context:
such as image recognition, object detection and more. 
 
3.Recurrent Neural Network (RNN) 
It is a type of neural network designed to handle sequential data like text, time series or speech. 
Unlike regular networks, RNNs have loops that allow them to carry information from previous steps 
to the current step, making them good at learning from context.  
They are commonly used in language modelling, sentiment analysis, and speech recognition. 
 
4.LSTM

4.LSTM 
LSTM (Long Short-Term Memory) is a special type of RNN designed to remember information for long 
periods and avoid the vanishing gradient problem. 
 LSTMs are widely used in applications like machine translation, text generation, and time series 
forecasting. 
 
5.Auto encoders 
It is a type of neural network that compresses input data into smaller more efficient representation 
while preserving the most important information. It has two parts encoder that co

#### Send to Gemini

In [32]:
response = model.generate_content(final_prompt)
print(response.text)

Recurrent Neural Network (RNN) is a type of neural network designed to handle sequential data like text, time series, or speech. Unlike regular networks, RNNs have loops that allow them to carry information from previous steps to the current step, making them good at learning from context. They are commonly used in language modeling, sentiment analysis, and speech recognition.
